# CAPM / MEDAF — Attijariwafa Bank vs MASI
**Auteur :** Mohamed El Otmany — FST Errachidia, Génie Financier & Actuariat  
**Objectif :** Estimation du bêta systématique (β) et de l'alpha (α) d'ATR via régression OLS, avec validation économétrique complète du modèle MEDAF.

---
**Modèle théorique :**
$$R_{i,t} - R_f = \alpha_i + \beta_i (R_{m,t} - R_f) + \varepsilon_{i,t}$$
où $\varepsilon_{i,t} \sim \mathcal{N}(0, \sigma^2)$ sous les hypothèses classiques de Gauss-Markov.

## 0. Configuration & Imports

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.stats.diagnostic import het_breuschpagan, acorr_ljungbox
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from scipy import stats
from src.utils import clean_investing_data, calculate_log_returns

# Aesthetic style
plt.style.use('seaborn-v0_8-whitegrid')
COLORS = {'stock': '#1f4e79', 'market': '#2e7d32', 'regression': '#c62828', 'neutral': '#546e7a'}

print('Libraries loaded successfully.')
print(f'Pandas: {pd.__version__} | NumPy: {np.__version__}')

## 1. Chargement & Prétraitement des Données

In [ ]:
# Chargement via la fonction utilitaire (gère les formats Investing.com)
stock_df  = clean_investing_data('../data/ATR.xlsx')   # Attijariwafa Bank
market_df = clean_investing_data('../data/MASI.xlsx')  # MASI

print('=== Attijariwafa Bank (ATR) ===')
print(f'Observations : {len(stock_df)} | Période : {stock_df.Date.min().date()} → {stock_df.Date.max().date()}')
print(stock_df.describe(), '\n')

print('=== MASI ===')
print(f'Observations : {len(market_df)} | Période : {market_df.Date.min().date()} → {market_df.Date.max().date()}')
print(market_df.describe())

In [ ]:
# Jointure sur les dates communes (intersection — exclut jours fériés asymétriques)
df = pd.merge(stock_df, market_df, on='Date', suffixes=('_stock', '_market'))

# Rendements log-normaux : r_t = ln(P_t / P_{t-1})
df['R_stock']  = calculate_log_returns(df['Price_stock'])
df['R_market'] = calculate_log_returns(df['Price_market'])
df = df.dropna().reset_index(drop=True)

print(f'Dataset final : {len(df)} observations communes')
print(f'Rendement moyen ATR  : {df.R_stock.mean()*100:.4f}%/jour')
print(f'Rendement moyen MASI : {df.R_market.mean()*100:.4f}%/jour')
df[['Date','Price_stock','Price_market','R_stock','R_market']].tail()

## 2. Analyse Exploratoire (EDA)

In [ ]:
fig = plt.figure(figsize=(16, 10))
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.4, wspace=0.3)

# --- Évolution des prix (normalisés base 100) ---
ax1 = fig.add_subplot(gs[0, :])
base_stock  = df['Price_stock'].iloc[0]
base_market = df['Price_market'].iloc[0]
ax1.plot(df['Date'], df['Price_stock']/base_stock*100,  color=COLORS['stock'],  lw=1.5, label='ATR (base 100)')
ax1.plot(df['Date'], df['Price_market']/base_market*100, color=COLORS['market'], lw=1.5, label='MASI (base 100)')
ax1.set_title('Évolution Comparée des Prix — ATR vs MASI (base 100)', fontsize=13, fontweight='bold')
ax1.legend()
ax1.set_ylabel('Indice de Prix')

# --- Distribution des rendements ---
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(df['R_stock'],  bins=60, alpha=0.6, color=COLORS['stock'],  density=True, label='ATR')
ax2.hist(df['R_market'], bins=60, alpha=0.6, color=COLORS['market'], density=True, label='MASI')
x_range = np.linspace(df['R_stock'].min(), df['R_stock'].max(), 200)
mu, sigma = df['R_stock'].mean(), df['R_stock'].std()
ax2.plot(x_range, stats.norm.pdf(x_range, mu, sigma), 'r--', lw=1.5, label='N(μ,σ²) théorique')
ax2.set_title('Distribution des Rendements Logarithmiques', fontsize=11, fontweight='bold')
ax2.legend(fontsize=9)
ax2.set_xlabel('Rendement log')

# --- Scatter Rendements ---
ax3 = fig.add_subplot(gs[1, 1])
ax3.scatter(df['R_market'], df['R_stock'], alpha=0.3, s=10, color=COLORS['neutral'])
ax3.axhline(0, color='k', lw=0.5, ls='--')
ax3.axvline(0, color='k', lw=0.5, ls='--')
ax3.set_xlabel('R_market (MASI)')
ax3.set_ylabel('R_stock (ATR)')
ax3.set_title('Nuage de Points : ATR vs MASI', fontsize=11, fontweight='bold')

plt.suptitle('Analyse Exploratoire — Données Boursières Marocaines', fontsize=14, y=1.01, fontweight='bold')
plt.savefig('../outputs/eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Estimation CAPM par MCO (OLS)

In [ ]:
# Spécification du modèle : R_i = alpha + beta * R_m + epsilon
# (Rf = 0 — approximation courante ou ajuster avec le taux des BdT marocains)
Y = df['R_stock']
X = sm.add_constant(df['R_market'])  # constante = alpha

model = sm.OLS(Y, X).fit()

alpha_hat = model.params['const']
beta_hat  = model.params['R_market']
r_squared = model.rsquared

print('=' * 60)
print('       RÉSULTATS DE L\'ESTIMATION CAPM (MCO)')
print('=' * 60)
print(model.summary())
print('\n--- INTERPRÉTATION ÉCONOMIQUE ---')
print(f'β = {beta_hat:.4f} → ', end='')
if beta_hat < 1:
    print(f'ATR est DÉFENSIVE — une variation de 1% du marché entraîne {beta_hat:.2f}% sur ATR.')
elif beta_hat > 1:
    print(f'ATR est AGRESSIVE — amplification du risque systématique.')
else:
    print('ATR réplique exactement le marché.')
print(f'α = {alpha_hat:.6f} → Surperformance anormale (CAPM test: H0: α=0, p={model.pvalues["const"]:.4f})')
print(f'R² = {r_squared:.4f} → {r_squared*100:.1f}% de la variance d\'ATR expliquée par le marché.')

In [ ]:
# --- Visualisation de la Security Market Line ---
fig, ax = plt.subplots(figsize=(10, 7))

ax.scatter(df['R_market'], df['R_stock'], alpha=0.35, s=12,
           color=COLORS['neutral'], label='Rendements quotidiens')

x_line = np.linspace(df['R_market'].min(), df['R_market'].max(), 300)
y_line = alpha_hat + beta_hat * x_line
ax.plot(x_line, y_line, color=COLORS['regression'], lw=2.5,
        label=f'SML : $R_{{ATR}} = {alpha_hat:.4f} + {beta_hat:.4f} \\cdot R_{{MASI}}$')

ax.axhline(0, color='k', lw=0.7, ls=':')
ax.axvline(0, color='k', lw=0.7, ls=':')

# Annotations
bbox_props = dict(boxstyle='round,pad=0.4', facecolor='lightyellow', alpha=0.8, edgecolor='gray')
stats_text = f'β = {beta_hat:.4f}\nα = {alpha_hat:.6f}\nR² = {r_squared:.4f}\nn = {len(df)}'
ax.text(0.05, 0.95, stats_text, transform=ax.transAxes, fontsize=10,
        verticalalignment='top', bbox=bbox_props, family='monospace')

ax.set_xlabel('Rendement MASI ($R_m$)', fontsize=12)
ax.set_ylabel('Rendement ATR ($R_i$)', fontsize=12)
ax.set_title('CAPM — Security Market Line\nAttijariwafa Bank vs MASI', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)

plt.tight_layout()
plt.savefig('../outputs/capm_sml.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Tests Économétriques de Validation

In [ ]:
residuals = model.resid

print('=' * 60)
print('   VALIDATION DES HYPOTHÈSES DE GAUSS-MARKOV')
print('=' * 60)

# ── 1. NORMALITÉ DES RÉSIDUS — Jarque-Bera & Shapiro-Wilk ──────────────────
jb_stat, jb_pval = stats.jarque_bera(residuals)
sw_stat, sw_pval = stats.shapiro(residuals[:5000])  # Shapiro limité à 5000 obs
kurt  = stats.kurtosis(residuals)
skew  = stats.skew(residuals)

print('\n[1] NORMALITÉ DES RÉSIDUS')
print(f'    Jarque-Bera : stat={jb_stat:.4f}, p={jb_pval:.4f}  → {"❌ Non-normale" if jb_pval<0.05 else "✅ Normale"}')
print(f'    Shapiro-Wilk: stat={sw_stat:.4f}, p={sw_pval:.4f}  → {"❌ Non-normale" if sw_pval<0.05 else "✅ Normale"}')
print(f'    Kurtosis    : {kurt:.4f} (excess) | Skewness : {skew:.4f}')
if abs(kurt) > 1:
    print('    ⚠️  Queues épaisses (leptokurtic) — typique des séries financières.')

# ── 2. AUTOCORRÉLATION — Durbin-Watson & Ljung-Box ──────────────────────────
dw_stat = durbin_watson(residuals)
lb_test = acorr_ljungbox(residuals, lags=[5, 10, 20], return_df=True)

print('\n[2] AUTOCORRÉLATION DES RÉSIDUS')
print(f'    Durbin-Watson: {dw_stat:.4f}  → {"✅ Pas d\'autocorrélation (≈2)" if 1.5<dw_stat<2.5 else "⚠️ Autocorrélation possible"}')
print('    Ljung-Box (lags 5, 10, 20):')
print(lb_test.to_string(index=True))

# ── 3. HÉTÉROSCÉDASTICITÉ — Breusch-Pagan & White ───────────────────────────
bp_lm, bp_pval, bp_fstat, bp_fpval = het_breuschpagan(residuals, model.model.exog)

print('\n[3] HOMOSCÉDASTICITÉ')
print(f'    Breusch-Pagan: LM={bp_lm:.4f}, p={bp_pval:.4f}  → {"⚠️ Hétéroscédasticité" if bp_pval<0.05 else "✅ Homoscédasticité"}')

# ── 4. MULTICOLINÉARITÉ (non pertinent ici mais structurant) ─────────────────
corr_xy = df[['R_stock','R_market']].corr().iloc[0,1]
print('\n[4] CORRÉLATION')
print(f'    Corrélation de Pearson (ATR, MASI) = {corr_xy:.4f}')

print('\n' + '=' * 60)
print('RÉSUMÉ DES HYPOTHÈSES')
print('=' * 60)

In [ ]:
# Visualisation des diagnostics de résidus
fig = plt.figure(figsize=(16, 12))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.45, wspace=0.35)

# 1. Résidus vs Fitted
ax1 = fig.add_subplot(gs[0, 0])
ax1.scatter(model.fittedvalues, residuals, alpha=0.3, s=8, color=COLORS['neutral'])
ax1.axhline(0, color=COLORS['regression'], lw=1.5, ls='--')
ax1.set_xlabel('Valeurs ajustées'); ax1.set_ylabel('Résidus')
ax1.set_title('Résidus vs Valeurs Ajustées', fontweight='bold')

# 2. Q-Q Plot
ax2 = fig.add_subplot(gs[0, 1])
stats.probplot(residuals, dist='norm', plot=ax2)
ax2.set_title('Q-Q Plot (Normalité)', fontweight='bold')
ax2.get_lines()[0].set(markersize=3, alpha=0.4, color=COLORS['neutral'])
ax2.get_lines()[1].set(color=COLORS['regression'])

# 3. Distribution des résidus
ax3 = fig.add_subplot(gs[0, 2])
ax3.hist(residuals, bins=60, density=True, color=COLORS['stock'], alpha=0.7, edgecolor='white')
x_kde = np.linspace(residuals.min(), residuals.max(), 300)
mu_r, sig_r = residuals.mean(), residuals.std()
ax3.plot(x_kde, stats.norm.pdf(x_kde, mu_r, sig_r), 'r--', lw=2, label='N(0,σ²)')
ax3.set_title('Distribution des Résidus', fontweight='bold')
ax3.legend()

# 4. ACF des résidus
ax4 = fig.add_subplot(gs[1, 0])
plot_acf(residuals, ax=ax4, lags=30, alpha=0.05, color=COLORS['stock'])
ax4.set_title('ACF des Résidus', fontweight='bold')

# 5. ACF des résidus au carré (effet ARCH)
ax5 = fig.add_subplot(gs[1, 1])
plot_acf(residuals**2, ax=ax5, lags=30, alpha=0.05, color=COLORS['market'])
ax5.set_title('ACF des Résidus² (Test ARCH)', fontweight='bold')

# 6. Scale-Location (homoscédasticité)
ax6 = fig.add_subplot(gs[1, 2])
ax6.scatter(model.fittedvalues, np.sqrt(np.abs(residuals)), alpha=0.3, s=8, color=COLORS['neutral'])
ax6.set_xlabel('Valeurs ajustées'); ax6.set_ylabel('√|Résidus|')
ax6.set_title('Scale-Location (Homoscédasticité)', fontweight='bold')

plt.suptitle('Diagnostics Économétriques — Validation des Hypothèses MCO', fontsize=13, fontweight='bold', y=1.01)
plt.savefig('../outputs/residual_diagnostics.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Décomposition du Risque

In [ ]:
# Décomposition variance : Var(R_i) = β²·Var(R_m) + Var(ε)
var_total      = df['R_stock'].var()
var_systematic = (beta_hat ** 2) * df['R_market'].var()
var_specific   = residuals.var()

pct_systematic = var_systematic / var_total * 100
pct_specific   = var_specific   / var_total * 100

print('=' * 55)
print('        DÉCOMPOSITION DU RISQUE (ATR)')
print('=' * 55)
print(f'Variance totale           : {var_total:.8f}')
print(f'  ├── Risque systématique  : {var_systematic:.8f}  ({pct_systematic:.1f}%)')
print(f'  └── Risque spécifique    : {var_specific:.8f}   ({pct_specific:.1f}%)')
print(f'\nVérification R² ≈ {pct_systematic/100:.4f} (vs R² OLS = {r_squared:.4f})')
print(f'\nVolatilité annualisée ATR  : {df.R_stock.std()*np.sqrt(252)*100:.2f}%')
print(f'Volatilité annualisée MASI : {df.R_market.std()*np.sqrt(252)*100:.2f}%')
print(f'Sharpe Ratio ATR (Rf=3.5%) : {((df.R_stock.mean()*252 - 0.035) / (df.R_stock.std()*np.sqrt(252))):.4f}')

# Pie Chart
fig, ax = plt.subplots(figsize=(7, 5))
sizes  = [pct_systematic, pct_specific]
labels = [f'Risque Systématique\n{pct_systematic:.1f}%', f'Risque Spécifique\n{pct_specific:.1f}%']
colors = [COLORS['stock'], COLORS['market']]
ax.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%',
       startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2), textprops={'fontsize': 11})
ax.set_title('Décomposition du Risque — ATR\n(Risque Systématique vs Idiosyncratique)', fontsize=12, fontweight='bold')
plt.savefig('../outputs/risk_decomposition.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 6. Synthèse & Conclusion

| Métrique | Valeur | Interprétation |
|---|---|---|
| **β (Bêta)** | *(cf. résultats)* | Risque systématique relatif au marché |
| **α (Alpha)** | *(cf. résultats)* | Surperformance anormale (H₀: α=0) |
| **R²** | *(cf. résultats)* | Part de variance expliquée par le MASI |
| **Jarque-Bera** | *(cf. tests)* | Normalité des résidus |
| **Durbin-Watson** | *(cf. tests)* | Absence d'autocorrélation |
| **Breusch-Pagan** | *(cf. tests)* | Homoscédasticité |

**Limites du modèle :**  
- Le CAPM suppose un seul facteur de risque (marché). Les modèles multi-facteurs (Fama-French 3/5 facteurs) permettent une meilleure explication.  
- Les queues épaisses observées (kurtosis > 3) suggèrent l'utilisation de modèles GARCH pour la modélisation de la volatilité conditionnelle.

**Extensions possibles :**  
- Rolling beta (fenêtre glissante) pour capturer la non-stationnarité de β  
- Modèle CAPM avec taux sans risque variable (BdT 52 semaines de BAM)  
- Comparaison avec d'autres valeurs du Casablanca Stock Exchange